# Volume 3. NEMO-Style Toy Parser

Question: how does the maintained parser wire together lexical categories, roles, and word order on a tiny sentence?

This is a toy pipeline over three words. It is useful for inspecting how assemblies move through parser stages, but it is not evidence of broad language understanding.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from neural_assemblies.assembly_calculus import NemoParser
from neural_assemblies.assembly_calculus.parser import ROLE_LABELS
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import plot_assemblies, plot_overlap_matrix


Register three word forms with category and grounding labels. The grounding labels are stimuli used by the toy learner; they are not images or motor programs.


In [ ]:
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
parser = NemoParser(brain, n=5_000, k=60, beta=0.10, rounds=6)
parser.setup_areas()

vocabulary = [
    {"word": "dog", "category": "noun", "grounding": "vis_dog"},
    {"word": "chases", "category": "verb", "grounding": "mot_chases"},
    {"word": "cat", "category": "noun", "grounding": "vis_cat"},
]
for item in vocabulary:
    parser.register_word(item["word"], item["category"], item["grounding"])

pd.DataFrame(vocabulary)


Training is staged so the notebook can inspect each layer: lexical category assemblies, role-bound assemblies, and word-order memory.


In [ ]:
training_sentences = [["dog", "chases", "cat"]]
parser.train_lexicon()
parser.train_roles(training_sentences)
parser.train_word_order(training_sentences)

pd.DataFrame([
    {"stage": "lexicon", "artifact": "noun_lexicon", "entries": len(parser.noun_lexicon)},
    {"stage": "lexicon", "artifact": "verb_lexicon", "entries": len(parser.verb_lexicon)},
    {
        "stage": "roles",
        "artifact": "role_lexicons",
        "entries": sum(len(lexicon) for lexicon in parser.role_lexicons.values()),
    },
    {"stage": "sequence", "artifact": "SEQ area", "entries": len(training_sentences[0])},
])


In [ ]:
lexical_rows = []
for word, assembly in parser.noun_lexicon.items():
    lexical_rows.append({
        "word": word,
        "category_area": assembly.area,
        "winner_count": len(assembly),
        "sample_winners": assembly.winners[:10].tolist(),
    })
for word, assembly in parser.verb_lexicon.items():
    lexical_rows.append({
        "word": word,
        "category_area": assembly.area,
        "winner_count": len(assembly),
        "sample_winners": assembly.winners[:10].tolist(),
    })
pd.DataFrame(lexical_rows)


In [ ]:
ax, matrix = plot_overlap_matrix(
    [parser.noun_lexicon["dog"], parser.noun_lexicon["cat"]],
    labels=["dog", "cat"],
    cmap="viridis",
)
ax.set_title("Noun lexicon overlap in LEX_NOUN")
plt.show()
plt.close(ax.figure)


Role assemblies live in separate role areas, so a cross-role overlap matrix would be easy to misread. A table and sparse maps are clearer for this first parser notebook.


In [ ]:
role_rows = []
role_assemblies = []
role_titles = []
for role_area, lexicon in parser.role_lexicons.items():
    for word, assembly in lexicon.items():
        role_rows.append({
            "role_area": role_area,
            "role_label": ROLE_LABELS[role_area],
            "word": word,
            "winner_count": len(assembly),
            "sample_winners": assembly.winners[:10].tolist(),
        })
        role_assemblies.append(assembly)
        role_titles.append(f"{ROLE_LABELS[role_area]}: {word}")

pd.DataFrame(role_rows)


In [ ]:
fig, axes = plot_assemblies(
    role_assemblies,
    parser.n,
    titles=role_titles,
    subtitles=["role-bound assembly" for _ in role_assemblies],
    colors=["#4d9de0", "#59a14f", "#e15554"],
    marker_size=16,
)
plt.show()
plt.close(fig)


Now parse a sentence with the nouns swapped. Because the parser has a simple SVO rule at this level, the first noun becomes AGENT and the second noun becomes PATIENT.


In [ ]:
sentence = ["cat", "chases", "dog"]
result = parser.parse(sentence)

pd.DataFrame([
    {
        "position": index + 1,
        "word": word,
        "category": result["categories"][word],
        "role": result["roles"][word],
    }
    for index, word in enumerate(sentence)
])


The useful takeaway is compositional bookkeeping: the parser keeps lexical assemblies, role assemblies, and order information as separate inspectable artifacts. The stronger language claims belong in later notebooks and research validations.
